# Live: Pharmacovigilance Signal Monitor

Pulls the most recent batch from the public openFDA drug adverse event
feed and renders a chart of the top reaction signals — a live snapshot
of a continuously-updated public feed, not canned/static data. Re-run
the last cell any time for a fresh snapshot. Pharma flavor of "watch a
live feed."

Pure Python + pandas + Matplotlib — no Spark session needed, since this
reads directly from the live API rather than a persisted table. For a
durable, queryable copy see `data-generators/fda_adverse_event_ingest.ipynb`.

Requires `pandas` and `matplotlib` — installed by the next cell.

In [ ]:
!pip3 install -q pandas matplotlib

In [ ]:
import json
import urllib.request
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd

## Configuration

Pulls the most recent 100 reports, sorted by receive date.

In [ ]:
API_URL = "https://api.fda.gov/drug/event.json?sort=receivedate:desc&limit=100"

## Fetch + flatten one snapshot

In [ ]:
def fetch_recent_reactions_df():
    with urllib.request.urlopen(API_URL) as resp:
        data = json.load(resp)

    rows = []
    for report in data.get("results", []):
        patient = report.get("patient", {})
        for reaction in patient.get("reaction", []):
            rows.append({
                "safety_report_id": report.get("safetyreportid"),
                "reaction_term": reaction.get("reactionmeddrapt"),
                "serious": report.get("serious") == "1",
            })
    return pd.DataFrame(rows)

## Fetch and render

Pulls the current snapshot and renders the top reaction terms in it.

In [ ]:
plt.style.use("dark_background")

reactions_df = fetch_recent_reactions_df()
top_reactions_df = (
    reactions_df.groupby("reaction_term")
    .agg(report_count=("safety_report_id", "count"), serious_count=("serious", "sum"))
    .reset_index()
    .sort_values("report_count", ascending=False)
    .head(15)
    .sort_values("report_count")
)
refreshed_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

norm = plt.Normalize(top_reactions_df["serious_count"].min(), top_reactions_df["serious_count"].max())
colors = plt.cm.OrRd(norm(top_reactions_df["serious_count"]))

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_reactions_df["reaction_term"], top_reactions_df["report_count"], color=colors)
ax.set_xlabel("report_count")
ax.set_title(f"Live Adverse Event Signals — top reactions in most recent 100 reports — as of {refreshed_at}")
sm = plt.cm.ScalarMappable(cmap="OrRd", norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="serious_count")
plt.tight_layout()
plt.show()

print(f"{len(reactions_df)} reaction records in this batch")